# CYP3A4 Inhibitor Drug Summary

This notebook summarizes CYP3A4 inhibitor exposure among MDD patients and among MDD + antidepressant patients.

The analysis calculates unique patient counts by CYP3A4 inhibitor drug across multiple observation windows, including overall exposure and selected recent time windows.


## Dataset A: MDD patients with CYP3A4 inhibitor exposure

The first dataset imports CYP3A4 inhibitor drug exposure records for the MDD cohort.


In [ ]:
import pandas
import os

# This query represents dataset "CYP Drug Summary for MDD patients" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_80298865_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1307863, 1309944, 1328165, 1703653, 1704139, 1714277, 1746940, 1748921, 1750500, 1754994, 42874220, 985708)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) d_exposure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
            ON d_exposure.drug_type_concept_id = d_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
            ON d_exposure.route_concept_id = d_route.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
            ON v.visit_concept_id = d_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
            ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_80298865_drug_df = pandas.read_gbq(
    dataset_80298865_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_80298865_drug_df.head(5)

## MDD condition records for Dataset A

These condition records are used to identify MDD diagnosis dates and to support date-window summaries.


In [ ]:
import pandas
import os

# This query represents dataset "CYP Drug Summary for MDD patients" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_80298865_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4152280)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) )
            )) c_occurrence 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_80298865_condition_df = pandas.read_gbq(
    dataset_80298865_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_80298865_condition_df.head(5)

## CYP3A4 inhibitor mapping and summary functions

This section defines the CYP3A4 inhibitor drug order, date-window logic, and patient-count summary functions.


In [ ]:
import pandas as pd
import numpy as np

# Keep the same order as your Excel table
cyp_drug_order = [
    "fluconazole",
    "ketoconazole",
    "erythromycin",
    "verapamil",
    "diltiazem",
    "ritonavir",
    "amiodarone",
    "clarithromycin",
    "cobicistat",
    "voriconazole",
    "itraconazole",
    "posaconazole"
]


def clean_cyp_drug_name(drug_name):
    """
    Convert standard_concept_name to ingredient-level clean name.
    Example:
    '100 ML fluconazole 2 MG/ML Injection' -> 'fluconazole'
    'verapamil hydrochloride 240 MG Extended Release' -> 'verapamil'
    """
    if pd.isna(drug_name):
        return np.nan
    
    name = str(drug_name).lower()
    
    for drug in cyp_drug_order:
        if drug in name:
            return drug
    
    return np.nan


def get_cyp_unique_patient_counts(
    condition_df,
    drug_df,
    start_date=None,
    end_date=None
):
    """
    Count unique patients by CYP drug name.

    Logic:
    1. Find first MDD diagnosis date for each person.
    2. Keep CYP medication records where drug_exposure_start_date >= first MDD diagnosis date.
    3. Optional: restrict CYP drug_exposure_start_date to a time window.
    4. Group by cleaned CYP drug name and count unique patients.

    Note:
    We use standard_concept_name to identify CYP ingredient names because
    drug_concept_id often represents specific formulations/dose forms, not the
    ingredient-level concept selected in the concept set.
    """
    
    condition_df = condition_df.copy()
    drug_df = drug_df.copy()
    
    # Convert datetime columns
    condition_df["condition_start_datetime"] = pd.to_datetime(
        condition_df["condition_start_datetime"], errors="coerce"
    )
    drug_df["drug_exposure_start_datetime"] = pd.to_datetime(
        drug_df["drug_exposure_start_datetime"], errors="coerce"
    )
    
    # Use DATE only, not full datetime
    condition_df["condition_start_date"] = condition_df["condition_start_datetime"].dt.date
    drug_df["drug_exposure_start_date"] = drug_df["drug_exposure_start_datetime"].dt.date
    
    # First MDD diagnosis date for each person
    first_mdd = (
        condition_df
        .dropna(subset=["condition_start_date"])
        .groupby("person_id", as_index=False)["condition_start_date"]
        .min()
        .rename(columns={"condition_start_date": "mdd_index_date"})
    )
    
    # Clean CYP drug name from standard_concept_name
    drug_df["drug_name_clean"] = drug_df["standard_concept_name"].apply(clean_cyp_drug_name)
    
    # Keep recognized CYP records only
    cyp_records = (
        drug_df
        .dropna(subset=["drug_name_clean", "drug_exposure_start_date"])
        [["person_id", "standard_concept_name", "drug_name_clean", "drug_exposure_start_date"]]
        .copy()
    )
    
    # Join CYP records with first MDD date
    joined = cyp_records.merge(first_mdd, on="person_id", how="inner")
    
    # Keep CYP medications on or after first MDD diagnosis date
    after_mdd = joined[
        joined["drug_exposure_start_date"] >= joined["mdd_index_date"]
    ].copy()
    
    # Optional time restriction based on CYP drug start date only
    if start_date is not None:
        start_date = pd.to_datetime(start_date).date()
        after_mdd = after_mdd[
            after_mdd["drug_exposure_start_date"] >= start_date
        ].copy()
    
    if end_date is not None:
        end_date = pd.to_datetime(end_date).date()
        after_mdd = after_mdd[
            after_mdd["drug_exposure_start_date"] <= end_date
        ].copy()
    
    # Count unique patients by CYP drug
    summary = (
        after_mdd
        .groupby("drug_name_clean", as_index=False)["person_id"]
        .nunique()
        .rename(columns={"person_id": "unique_patient_count"})
    )
    
    # Keep Excel order and include drugs with 0 count
    summary = (
        pd.DataFrame({"drug_name_clean": cyp_drug_order})
        .merge(summary, on="drug_name_clean", how="left")
    )
    
    summary["unique_patient_count"] = (
        summary["unique_patient_count"]
        .fillna(0)
        .astype(int)
    )
    
    # Add total unique patients row
    total_unique_patients = after_mdd["person_id"].nunique()
    
    total_row = pd.DataFrame({
        "drug_name_clean": ["total unique patients"],
        "unique_patient_count": [total_unique_patients]
    })
    
    summary = pd.concat([summary, total_row], ignore_index=True)
    
    return summary, after_mdd

In [ ]:
def check_dataset_latest_dates(condition_df=None, drug_df=None):
    """
    Check the latest available dates in the exported dataset.

    For condition table:
    - latest condition_start_datetime
    - latest condition_end_datetime

    For drug table:
    - latest drug_exposure_start_datetime
    - latest drug_exposure_end_datetime

    This helps identify the latest observed record date in the dataset.
    """
    
    results = []

    if condition_df is not None:
        condition_df = condition_df.copy()
        
        if "condition_start_datetime" in condition_df.columns:
            condition_df["condition_start_datetime"] = pd.to_datetime(
                condition_df["condition_start_datetime"], errors="coerce", utc=True
            )
            results.append({
                "table": "condition",
                "date_field": "condition_start_datetime",
                "latest_date": condition_df["condition_start_datetime"].max()
            })
        
        if "condition_end_datetime" in condition_df.columns:
            condition_df["condition_end_datetime"] = pd.to_datetime(
                condition_df["condition_end_datetime"], errors="coerce", utc=True
            )
            results.append({
                "table": "condition",
                "date_field": "condition_end_datetime",
                "latest_date": condition_df["condition_end_datetime"].max()
            })

    if drug_df is not None:
        drug_df = drug_df.copy()
        
        if "drug_exposure_start_datetime" in drug_df.columns:
            drug_df["drug_exposure_start_datetime"] = pd.to_datetime(
                drug_df["drug_exposure_start_datetime"], errors="coerce", utc=True
            )
            results.append({
                "table": "drug",
                "date_field": "drug_exposure_start_datetime",
                "latest_date": drug_df["drug_exposure_start_datetime"].max()
            })
        
        if "drug_exposure_end_datetime" in drug_df.columns:
            drug_df["drug_exposure_end_datetime"] = pd.to_datetime(
                drug_df["drug_exposure_end_datetime"], errors="coerce", utc=True
            )
            results.append({
                "table": "drug",
                "date_field": "drug_exposure_end_datetime",
                "latest_date": drug_df["drug_exposure_end_datetime"].max()
            })

    latest_dates_df = pd.DataFrame(results)
    
    if len(latest_dates_df) > 0:
        latest_dates_df["latest_date_only"] = latest_dates_df["latest_date"].dt.date
    
    return latest_dates_df
latest_dates_mdd_cyp = check_dataset_latest_dates(

    condition_df=dataset_80298865_condition_df,

    drug_df=dataset_80298865_drug_df

)

latest_dates_mdd_cyp

In [ ]:
def sort_summary_keep_total(summary_df):
    drug_rows = summary_df[
        summary_df["drug_name_clean"] != "total unique patients"
    ].copy()
    
    total_row = summary_df[
        summary_df["drug_name_clean"] == "total unique patients"
    ].copy()
    
    drug_rows_sorted = (
        drug_rows
        .sort_values("unique_patient_count", ascending=False)
        .reset_index(drop=True)
    )
    
    summary_sorted = pd.concat(
        [drug_rows_sorted, total_row],
        ignore_index=True
    )
    
    return summary_sorted

## Dataset A summaries

The following cells generate CYP3A4 inhibitor patient-count summaries for overall exposure and selected calendar windows.


In [ ]:
tab_a_summary, tab_a_records = get_cyp_unique_patient_counts(
    condition_df=dataset_80298865_condition_df,
    drug_df=dataset_80298865_drug_df
)

tab_a_summary_sorted = sort_summary_keep_total(tab_a_summary)
tab_a_summary_sorted

In [ ]:
# Tab B: 11/1/2022 to 10/31/2023
tab_b_summary, tab_b_records = get_cyp_unique_patient_counts(

    condition_df=dataset_80298865_condition_df,

    drug_df=dataset_80298865_drug_df,

    start_date="2022-11-01",

    end_date="2023-10-31"

)

tab_b_summary_sorted = sort_summary_keep_total(tab_b_summary)

tab_b_summary_sorted

In [ ]:
# Tab C: 5/1/2023 to 10/31/2023
tab_c_summary, tab_c_records = get_cyp_unique_patient_counts(

    condition_df=dataset_80298865_condition_df,

    drug_df=dataset_80298865_drug_df,

    start_date="2023-05-01",

    end_date="2023-10-31"

)

tab_c_summary_sorted = sort_summary_keep_total(tab_c_summary)

tab_c_summary_sorted

## Dataset B: MDD + antidepressant patients with CYP3A4 inhibitor exposure

The second dataset repeats the CYP3A4 inhibitor summary within the MDD + antidepressant cohort.


In [ ]:
import pandas
import os

# This query represents dataset "CYP Drug Summary for MDD + antidepressant patients" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_88633518_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4152280)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (798834, 35603277, 715939, 766209, 713109, 46275300, 721724, 781705, 755695, 754270, 738156, 715259, 739138, 1510996, 722031, 37498659, 797617, 778268, 43560354, 743670, 750982, 733896, 751412, 710062, 703547, 705755, 725131, 40234834, 703470, 716968, 44507700, 1366610, 714684, 717607)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) c_occurrence 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                ON c_occurrence.condition_type_concept_id = c_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                ON v.visit_concept_id = visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_88633518_condition_df = pandas.read_gbq(
    dataset_88633518_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_88633518_condition_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "CYP Drug Summary for MDD + antidepressant patients" for domain "drug" and was generated for All of Us Controlled Tier Dataset v8
dataset_88633518_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1307863, 1309944, 1328165, 1703653, 1704139, 1714277, 1746940, 1748921, 1750500, 1754994, 42874220, 985708)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (4152280)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 )) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (798834, 35603277, 715939, 766209, 713109, 46275300, 721724, 781705, 755695, 754270, 738156, 715259, 739138, 1510996, 722031, 37498659, 797617, 778268, 43560354, 743670, 750982, 733896, 751412, 710062, 703547, 705755, 725131, 40234834, 703470, 716968, 44507700, 1366610, 714684, 717607)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) ))) d_exposure 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                ON d_exposure.drug_type_concept_id = d_type.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                ON d_exposure.route_concept_id = d_route.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                ON v.visit_concept_id = d_visit.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_88633518_drug_df = pandas.read_gbq(
    dataset_88633518_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_88633518_drug_df.head(5)

## CYP3A4 summaries for MDD + antidepressant cohort

This section generates patient-count summaries by CYP3A4 inhibitor drug for the MDD + antidepressant cohort.


In [ ]:
import pandas as pd
import numpy as np

# CYP drugs to identify from standard_concept_name
cyp_drug_order = [
    "fluconazole",
    "ketoconazole",
    "erythromycin",
    "verapamil",
    "diltiazem",
    "ritonavir",
    "amiodarone",
    "clarithromycin",
    "cobicistat",
    "voriconazole",
    "itraconazole",
    "posaconazole"
]


def clean_cyp_drug_name(drug_name):
    """
    Convert standard_concept_name to ingredient-level clean CYP drug name.
    Example:
    '100 ML fluconazole 2 MG/ML Injection' -> 'fluconazole'
    """
    if pd.isna(drug_name):
        return np.nan
    
    name = str(drug_name).lower()
    
    for drug in cyp_drug_order:
        if drug in name:
            return drug
    
    return np.nan


def get_cyp_unique_patient_counts_sorted(
    condition_df,
    drug_df,
    start_date=None,
    end_date=None
):
    """
    Count unique patients by CYP drug name.

    Logic:
    1. Find each person's first MDD diagnosis date.
    2. Keep CYP medication records with drug_exposure_start_date >= first MDD diagnosis date.
    3. Optionally restrict CYP drug_exposure_start_date to a time window.
    4. Count distinct patients for each CYP drug.
    5. Sort by unique_patient_count from high to low.

    Note:
    Percent of total is not calculated here. Use Excel:
    unique_patient_count / 68301
    """
    
    condition_df = condition_df.copy()
    drug_df = drug_df.copy()
    
    # Convert datetime columns
    condition_df["condition_start_datetime"] = pd.to_datetime(
        condition_df["condition_start_datetime"], errors="coerce"
    )
    drug_df["drug_exposure_start_datetime"] = pd.to_datetime(
        drug_df["drug_exposure_start_datetime"], errors="coerce"
    )
    
    # Compare DATE only, not full datetime
    condition_df["condition_start_date"] = condition_df["condition_start_datetime"].dt.date
    drug_df["drug_exposure_start_date"] = drug_df["drug_exposure_start_datetime"].dt.date
    
    # First MDD diagnosis date per person
    first_mdd = (
        condition_df
        .dropna(subset=["condition_start_date"])
        .groupby("person_id", as_index=False)["condition_start_date"]
        .min()
        .rename(columns={"condition_start_date": "mdd_index_date"})
    )
    
    # Clean CYP drug name
    drug_df["drug_name_clean"] = drug_df["standard_concept_name"].apply(clean_cyp_drug_name)
    
    # Keep recognized CYP drug records
    cyp_records = (
        drug_df
        .dropna(subset=["drug_name_clean", "drug_exposure_start_date"])
        [["person_id", "standard_concept_name", "drug_name_clean", "drug_exposure_start_date"]]
        .copy()
    )
    
    # Join CYP records with first MDD date
    joined = cyp_records.merge(first_mdd, on="person_id", how="inner")
    
    # Keep CYP medications on or after first MDD diagnosis date
    after_mdd = joined[
        joined["drug_exposure_start_date"] >= joined["mdd_index_date"]
    ].copy()
    
    # Optional time restriction based on CYP drug start date only
    if start_date is not None:
        start_date = pd.to_datetime(start_date).date()
        after_mdd = after_mdd[
            after_mdd["drug_exposure_start_date"] >= start_date
        ].copy()
    
    if end_date is not None:
        end_date = pd.to_datetime(end_date).date()
        after_mdd = after_mdd[
            after_mdd["drug_exposure_start_date"] <= end_date
        ].copy()
    
    # Unique patient count by CYP drug
    summary = (
        after_mdd
        .groupby("drug_name_clean", as_index=False)["person_id"]
        .nunique()
        .rename(columns={"person_id": "unique_patient_count"})
    )
    
    # Add missing CYP drugs as 0
    summary = (
        pd.DataFrame({"drug_name_clean": cyp_drug_order})
        .merge(summary, on="drug_name_clean", how="left")
    )
    
    summary["unique_patient_count"] = (
        summary["unique_patient_count"]
        .fillna(0)
        .astype(int)
    )
    
    # Sort from high to low
    summary = (
        summary
        .sort_values("unique_patient_count", ascending=False)
        .reset_index(drop=True)
    )
    
    # Total unique patients row
    total_unique_patients = after_mdd["person_id"].nunique()
    
    total_row = pd.DataFrame({
        "drug_name_clean": ["total unique patients"],
        "unique_patient_count": [total_unique_patients]
    })
    
    summary = pd.concat([summary, total_row], ignore_index=True)
    
    return summary, after_mdd

In [ ]:
type_a_mdd_ad_cyp_summary, type_a_mdd_ad_cyp_records = get_cyp_unique_patient_counts_sorted(
    condition_df=dataset_88633518_condition_df,
    drug_df=dataset_88633518_drug_df
)

type_a_mdd_ad_cyp_summary

In [ ]:
type_b_mdd_ad_cyp_summary, type_b_mdd_ad_cyp_records = get_cyp_unique_patient_counts_sorted(
    condition_df=dataset_88633518_condition_df,
    drug_df=dataset_88633518_drug_df,
    start_date="2022-11-01",
    end_date="2023-10-31"
)

type_b_mdd_ad_cyp_summary

In [ ]:
type_c_mdd_ad_cyp_summary, type_c_mdd_ad_cyp_records = get_cyp_unique_patient_counts_sorted(
    condition_df=dataset_88633518_condition_df,
    drug_df=dataset_88633518_drug_df,
    start_date="2023-05-01",
    end_date="2023-10-31"
)

type_c_mdd_ad_cyp_summary